In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

C:\Users\Darshan V\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [3]:
df = pd.read_csv("../../data/processed/delays/delays_processed.csv")

df["FL_DATE"] = pd.to_datetime(df["FL_DATE"])

df.head()

,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT
0,2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,1562,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",1155,1151.0,-4.0,19.0,1210.0,1443.0,4.0,1501,1447.0,-14.0,0.0,NaN,0.0,186.0,176.0,153.0,1065.0,0.0,0.0,0.0,0.0,0.0
1,2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1149,MSP,"Minneapolis, MN",SEA,"Seattle, WA",2120,2114.0,-6.0,9.0,2123.0,2232.0,38.0,2315,2310.0,-5.0,0.0,NaN,0.0,235.0,236.0,189.0,1399.0,0.0,0.0,0.0,0.0,0.0
2,2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,459,DEN,"Denver, CO",MSP,"Minneapolis, MN",954,1000.0,6.0,20.0,1020.0,1247.0,5.0,1252,1252.0,0.0,0.0,NaN,0.0,118.0,112.0,87.0,680.0,0.0,0.0,0.0,0.0,0.0
3,2023-03-06,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,2295,MSP,"Minneapolis, MN",SFO,"San Francisco, CA",1609,1608.0,-1.0,27.0,1635.0,1844.0,9.0,1829,1853.0,24.0,0.0,NaN,0.0,260.0,285.0,249.0,1589.0,0.0,0.0,24.0,0.0,0.0
4,2020-02-23,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,407,MCO,"Orlando, FL",DFW,"Dallas/Fort Worth, TX",1840,1838.0,-2.0,15.0,1853.0,2026.0,14.0,2041,2040.0,-1.0,0.0,NaN,0.0,181.0,182.0,153.0,985.0,0.0,0.0,0.0,0.0,0.0


In [4]:
df["TOTAL_DELAY"] = (
    df["DEP_DELAY"].fillna(0)
    +
    df["ARR_DELAY"].fillna(0)
)

In [5]:
def delay_severity(delay):

    if delay <= 0:
        return "No Delay"

    elif delay <= 15:
        return "Minor"

    elif delay <= 60:
        return "Moderate"

    elif delay <= 180:
        return "Major"

    else:
        return "Severe"

df["DELAY_SEVERITY"] = df["ARR_DELAY"].fillna(0).apply(delay_severity)

In [6]:
df["DELAY_SEVERITY"].value_counts()

DELAY_SEVERITY
No Delay    2021853
Minor        462858
Moderate     340154
Major        142872
Severe        32263
Name: count, dtype: int64

In [7]:
def delay_category(delay):

    if delay < -5:
        return "Early"

    elif delay <= 15:
        return "On Time"

    else:
        return "Delayed"

df["DELAY_CATEGORY"] = df["ARR_DELAY"].fillna(0).apply(delay_category)

In [8]:
df["FLIGHT_PERFORMANCE_INDEX"] = (
    100
    - df["ARR_DELAY"].fillna(0).clip(lower=0)
)

df["FLIGHT_PERFORMANCE_INDEX"] = (
    df["FLIGHT_PERFORMANCE_INDEX"]
    .clip(lower=0)
)

In [9]:
df["OPERATIONAL_DISRUPTION"] = np.where(

    (df["CANCELLED"] == 1)
    |
    (df["DIVERTED"] == 1)
    |
    (df["ARR_DELAY"] > 60),

    1,

    0

)

In [10]:
airline_avg_delay = (
    df.groupby("AIRLINE")["ARR_DELAY"]
      .mean()
)

In [11]:
def airline_category(delay):

    if delay <= 5:
        return "High"

    elif delay <= 15:
        return "Medium"

    else:
        return "Low"

airline_reliability = airline_avg_delay.apply(airline_category)

In [12]:
df["AIRLINE_RELIABILITY"] = (
    df["AIRLINE"]
    .map(airline_reliability)
)

In [13]:
airport_delay = (
    df.groupby("ORIGIN")["DEP_DELAY"]
      .mean()
)

In [14]:
def airport_category(delay):

    if delay <= 5:
        return "Low Delay"

    elif delay <= 15:
        return "Medium Delay"

    else:
        return "High Delay"

airport_delay_category = airport_delay.apply(airport_category)

df["AIRPORT_DELAY_CATEGORY"] = (
    df["ORIGIN"]
      .map(airport_delay_category)
)

In [15]:
def season(month):

    if month in [12,1,2]:
        return "Winter"

    elif month in [3,4,5]:
        return "Spring"

    elif month in [6,7,8]:
        return "Summer"

    else:
        return "Autumn"

df["SEASON"] = (
    df["FL_DATE"]
      .dt.month
      .apply(season)
)

In [16]:
hour = (
    df["CRS_DEP_TIME"]
    //100
)

def time_period(h):

    if 5 <= h < 12:
        return "Morning"

    elif 12 <= h < 17:
        return "Afternoon"

    elif 17 <= h < 21:
        return "Evening"

    else:
        return "Night"

df["TIME_OF_DAY"] = hour.apply(time_period)

In [17]:
new_features = [
    "TOTAL_DELAY",
    "DELAY_SEVERITY",
    "DELAY_CATEGORY",
    "FLIGHT_PERFORMANCE_INDEX",
    "OPERATIONAL_DISRUPTION",
    "AIRLINE_RELIABILITY",
    "AIRPORT_DELAY_CATEGORY",
    "SEASON",
    "TIME_OF_DAY"
]

df[new_features].head()

,TOTAL_DELAY,DELAY_SEVERITY,DELAY_CATEGORY,FLIGHT_PERFORMANCE_INDEX,OPERATIONAL_DISRUPTION,AIRLINE_RELIABILITY,AIRPORT_DELAY_CATEGORY,SEASON,TIME_OF_DAY
0,-18.0,No Delay,Early,100.0,0,Medium,Medium Delay,Winter,Morning
1,-11.0,No Delay,On Time,100.0,0,High,Medium Delay,Autumn,Night
2,6.0,No Delay,On Time,100.0,0,Medium,Medium Delay,Summer,Morning
3,23.0,Moderate,Delayed,76.0,0,High,Medium Delay,Spring,Afternoon
4,-3.0,No Delay,On Time,100.0,0,Medium,Medium Delay,Winter,Evening


In [18]:
df.to_csv(
    "../../data/processed/delays/delays_feature_engineered.csv",
    index=False
)

print("Feature engineered dataset saved successfully.")

Feature engineered dataset saved successfully.


In [19]:
print(df.columns.tolist())
print(df.shape)
df.head()

['FL_DATE', 'AIRLINE', 'AIRLINE_DOT', 'AIRLINE_CODE', 'DOT_CODE', 'FL_NUMBER', 'ORIGIN', 'ORIGIN_CITY', 'DEST', 'DEST_CITY', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT', 'TOTAL_DELAY', 'DELAY_SEVERITY', 'DELAY_CATEGORY', 'FLIGHT_PERFORMANCE_INDEX', 'OPERATIONAL_DISRUPTION', 'AIRLINE_RELIABILITY', 'AIRPORT_DELAY_CATEGORY', 'SEASON', 'TIME_OF_DAY']
(3000000, 41)


,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT,TOTAL_DELAY,DELAY_SEVERITY,DELAY_CATEGORY,FLIGHT_PERFORMANCE_INDEX,OPERATIONAL_DISRUPTION,AIRLINE_RELIABILITY,AIRPORT_DELAY_CATEGORY,SEASON,TIME_OF_DAY
0,2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,1562,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",1155,1151.0,-4.0,19.0,1210.0,1443.0,4.0,1501,1447.0,-14.0,0.0,NaN,0.0,186.0,176.0,153.0,1065.0,0.0,0.0,0.0,0.0,0.0,-18.0,No Delay,Early,100.0,0,Medium,Medium Delay,Winter,Morning
1,2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1149,MSP,"Minneapolis, MN",SEA,"Seattle, WA",2120,2114.0,-6.0,9.0,2123.0,2232.0,38.0,2315,2310.0,-5.0,0.0,NaN,0.0,235.0,236.0,189.0,1399.0,0.0,0.0,0.0,0.0,0.0,-11.0,No Delay,On Time,100.0,0,High,Medium Delay,Autumn,Night
2,2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,459,DEN,"Denver, CO",MSP,"Minneapolis, MN",954,1000.0,6.0,20.0,1020.0,1247.0,5.0,1252,1252.0,0.0,0.0,NaN,0.0,118.0,112.0,87.0,680.0,0.0,0.0,0.0,0.0,0.0,6.0,No Delay,On Time,100.0,0,Medium,Medium Delay,Summer,Morning
3,2023-03-06,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,2295,MSP,"Minneapolis, MN",SFO,"San Francisco, CA",1609,1608.0,-1.0,27.0,1635.0,1844.0,9.0,1829,1853.0,24.0,0.0,NaN,0.0,260.0,285.0,249.0,1589.0,0.0,0.0,24.0,0.0,0.0,23.0,Moderate,Delayed,76.0,0,High,Medium Delay,Spring,Afternoon
4,2020-02-23,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,407,MCO,"Orlando, FL",DFW,"Dallas/Fort Worth, TX",1840,1838.0,-2.0,15.0,1853.0,2026.0,14.0,2041,2040.0,-1.0,0.0,NaN,0.0,181.0,182.0,153.0,985.0,0.0,0.0,0.0,0.0,0.0,-3.0,No Delay,On Time,100.0,0,Medium,Medium Delay,Winter,Evening
